# RMShell Solve/Postprocess Tutorial

This tutorial shows the new RMShell API:

1. Build a shell model.
2. Create canonical material and load inputs.
3. Solve for the structural displacement state.
4. Postprocess outputs from either the solved state or an externally supplied displacement field.

In [ ]:
import numpy as np
import csdl_alpha as csdl
import dolfinx
import dolfinx.io
import meshio
from mpi4py import MPI

from femo_alpha.rm_shell.rm_shell_model import RMShellModel

In [ ]:
def clamped_boundary(x):
    return np.less(x[0], 1.0e-12)

nx, ny = 4, 2
xs = np.linspace(0.0, 10.0, nx + 1)
ys = np.linspace(0.0, 2.0, ny + 1)
points = np.array([[x, y, 0.0] for y in ys for x in xs], dtype=float)
cells = []
for j in range(ny):
    for i in range(nx):
        n0 = j * (nx + 1) + i
        n1 = n0 + 1
        n3 = n0 + (nx + 1)
        n2 = n3 + 1
        cells.append([n0, n1, n2, n3])

mesh_path = "./rmshell_tutorial_mesh.xdmf"
meshio.write(mesh_path, meshio.Mesh(points, [("quad", np.array(cells, dtype=np.int64))]))
with dolfinx.io.XDMFFile(MPI.COMM_WORLD, mesh_path, "r") as xdmf:
    mesh = xdmf.read_mesh(name="Grid")

shell = RMShellModel(
    mesh,
    shell_bc_func=clamped_boundary,
    element_wise_material=False,
    solve_direct=True,
    record=False,
)

In [ ]:
nn = shell.nn
thickness = csdl.Variable(value=0.1 * np.ones(nn), name="thickness")
E = csdl.Variable(value=1.0e8 * np.ones(nn), name="E")
nu = csdl.Variable(value=0.3 * np.ones(nn), name="nu")
density = csdl.Variable(value=10.0 * np.ones(nn), name="density")
nodal_pressure = csdl.Variable(value=np.zeros((nn, 3)), name="nodal_pressure")
nodal_pressure.value[:, 2] = 5.0
node_disp = csdl.Variable(value=np.zeros((nn, 3)), name="node_disp")

material = shell.material_inputs.from_isotropic(
    E=E,
    nu=nu,
    thickness=thickness,
    density=density,
)
loads = shell.load_inputs.from_fields(
    nodal_pressure=nodal_pressure,
    node_disp=node_disp,
)

In [ ]:
recorder = csdl.Recorder(inline=True)
recorder.start()

state = shell.solve(material=material, loads=loads)
outputs = shell.post.evaluate(state=state)

print("Compliance:", outputs.compliance.value)
print("Mass:", outputs.mass.value)
print("Tip deflection:", np.max(outputs.disp_extracted.value[:, 2]))

In [ ]:
# Evaluate postprocessing again with the same displacement supplied explicitly.
outputs_external = shell.post.evaluate(
    material=material,
    loads=loads,
    displacement=state.disp_solid,
)

print("External postprocess compliance:", outputs_external.compliance.value)

## Direct generalized load vectors

If an upstream transfer already provides the generalized shell load vector, skip field reconstruction:

```python
load_vector = shell.assemble_generalized_load_vector(
    nodal_pressure=nodal_pressure.value,
    node_disp=node_disp.value,
)
loads = shell.load_inputs.from_vector(load_vector=load_vector, node_disp=node_disp)
state = shell.solve(material=material, loads=loads)
outputs = shell.post.evaluate(state=state)
```